# Réseau de neurones simple avec Keras

Dans ces travaux pratiques, nous allons construire avec Keras des réseaux de neurones qui reconnaissent les chiffres manuscrits de MNIST :

1. une régression logistique multinomiale, le réseau sans couche cachée ;
2. un perceptron multicouche régularisé ;
3. une recherche d'hyper-paramètres avec KerasTuner ;
4. le suivi de l'entraînement avec TensorBoard, l'arrêt précoce et la sauvegarde du meilleur modèle.

Il n'est pas nécessaire d'utiliser une carte graphique pour ce TP : tout est rapide sur CPU. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q keras-tuner
import keras
import keras_tuner
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Récupération des données

Cette fois, nous récupérons le jeu de données par Keras.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Regardons les données

Étudiez comment sont stockées les données :

- la forme et le type des quatre tableaux ;
- la valeur des premières étiquettes et l'intervalle des pixels ;
- quelques images, avec leur étiquette ;
- le nombre d'exemples de chaque classe : les classes sont-elles équilibrées ?

**Documentation utile**

- [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les attributs et les appels à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()` et `X_train.max()`
- `np.bincount(y_train)` : le nombre d'occurrences de chaque entier, de 0 à 9
- `plt.subplots(1, 10)` renvoie une figure et 10 axes ; `ax.imshow(image, cmap="gray_r")` affiche une image 28 × 28 en niveaux de gris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(X_train.dtype, X_train.min(), X_train.max(), y_train[:10])
print(np.bincount(...))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(...)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
print(f"Format de X_train : {X_train.shape}, de X_test : {X_test.shape}")
print(f"Format de y_train : {y_train.shape}, de y_test : {y_test.shape}")
print(f"Type des images : {X_train.dtype}, de {X_train.min()} à {X_train.max()}")
print(f"Type des étiquettes : {y_train.dtype}, premières : {y_train[:10]}")
print(f"Exemples par classe : {np.bincount(y_train)}")

# 60 000 images d'entraînement et 10 000 de test, de 28 × 28 pixels ; des entiers
# non signés sur 8 bits (uint8), de 0 à 255 ; des étiquettes entières, de 0 à 9 ;
# de 5 421 à 6 742 exemples par classe : des classes à peu près équilibrées

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(y_train[i])
  ax.axis("off")
plt.show()

## Transformation des données

Nous devons transformer ces données :

1. Aplatir les images : un tableau de forme `(_, 784)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples. Chaque image devient un vecteur de 784 pixels, l'entrée attendue par une couche `Dense`.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt que $[0, 255]$.
3. Encoder les classes en [*one-hot*](https://fr.wikipedia.org/wiki/Encodage_one-hot) : la première des deux pertes que nous verrons, `categorical_crossentropy`, attend pour chaque exemple 10 valeurs, toutes nulles sauf un 1 à l'indice de sa classe.

Effectuez les deux premières transformations sur `X_train` et `X_test`. Pour la troisième, gardez `y_train` et `y_test`, et rangez les classes encodées dans `Y_train` et `Y_test`.

**Documentation utile**

- [`keras.utils.to_categorical`](https://keras.io/api/utils/python_utils/#tocategorical-function)

In [ ]:
nb_classes = 10
input_dim = 28 * 28

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.reshape(-1, input_dim)` : `-1` laisse NumPy calculer le nombre d'exemples
- `/ 255` divise chaque pixel et donne des flottants
- `keras.utils.to_categorical(y_train, num_classes=nb_classes)` renvoie un tableau de forme `(60000, 10)` ; l'étiquette 5 y devient `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = X_train.reshape(...) / ...
X_test = ...
Y_train = keras.utils.to_categorical(..., num_classes=nb_classes)
Y_test = ...
```

### Solution

In [ ]:
# Des images « à plat » : l'entrée du réseau est un vecteur de input_dim pixels,
# dans [0, 1]. -1 laisse NumPy calculer le nombre d'exemples.
X_train = X_train.reshape(-1, input_dim) / 255
X_test = X_test.reshape(-1, input_dim) / 255

# Les classes en vecteurs one-hot, pour categorical_crossentropy
Y_train = keras.utils.to_categorical(y_train, num_classes=nb_classes)
Y_test = keras.utils.to_categorical(y_test, num_classes=nb_classes)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)
print(f"Étiquette {y_train[0]} en one-hot : {Y_train[0]}")

## Régression logistique multinomiale

Créez avec `keras.Sequential` un modèle sans couche cachée, `model` : une entrée `keras.Input` de `input_dim` pixels, puis une couche `layers.Dense` de `nb_classes` neurones, avec l'activation softmax. Chaque neurone calcule une combinaison linéaire des pixels, et la softmax en fait 10 probabilités de somme 1 : c'est une régression logistique multinomiale.

Affichez son résumé avec `summary()` et expliquez son nombre de paramètres.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Sequential([...])` prend la liste des couches, dans l'ordre
- `keras.Input(shape=(input_dim,))` : la forme d'**un** exemple, sans la taille du batch
- `layers.Dense(nb_classes, activation="softmax")`
- Résultat attendu : 7 850 paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model = keras.Sequential([keras.Input(shape=...), layers.Dense(..., activation=...)])
model.summary()
```

### Solution

In [ ]:
model = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model.summary()

# Une couche Dense a un poids par entrée et par neurone, plus un biais par neurone :
# 784 × 10 + 10 = 7 850 paramètres. None est la taille du batch, libre.

## Apprentissage

Compilez le modèle avec `compile` : l'optimiseur `"sgd"` (descente de gradient stochastique, pas d'apprentissage de 0,01), la perte `categorical_crossentropy`, qui prend les cibles one-hot `Y_train`, et la métrique `"accuracy"`. Entraînez-le ensuite avec `fit`, avec :

- 128 exemples par batch ;
- 10 epochs ;
- 20 % des données d'entraînement en validation (`validation_split`).

Enfin, mesurez sa perte et son accuracy sur le jeu de test avec la fonction `evaluate` ci-dessous. Gardez ces chiffres : ils servent de référence aux modèles suivants.

In [ ]:
def evaluate(model: keras.Model, one_hot: bool = False) -> None:
  """Affiche la perte et l'accuracy de `model` sur le jeu de test.

  one_hot : True pour un modèle entraîné sur les cibles one-hot (Y_train).
  """
  loss, accuracy = model.evaluate(X_test, Y_test if one_hot else y_test, verbose=0)
  print(f"Test : perte {loss:.3f}, accuracy {accuracy:.3f}")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])`
- `model.fit(X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2)` renvoie un objet `History`, dont l'attribut `history` donne la perte et les métriques de chaque epoch
- `evaluate(model, one_hot=True)` : le modèle apprend sur des cibles one-hot
- Résultat attendu : environ 89 % d'accuracy en validation comme en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model.compile(optimizer=..., loss=..., metrics=[...])
history = model.fit(X_train, ..., batch_size=..., epochs=..., validation_split=...)
evaluate(model, one_hot=True)
```

### Solution

In [ ]:
# compile associe au modèle un optimiseur, une fonction de perte et des métriques
model.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

# fit entraîne le modèle compilé : une boucle d'entraînement complète
history = model.fit(X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(model, one_hot=True)

## Une perte pour les étiquettes entières

La perte `sparse_categorical_crossentropy` calcule la même entropie croisée, mais à partir des étiquettes entières `y_train` : l'encodage one-hot devient inutile.

Créez un modèle `model2` identique au précédent, compilez-le avec cette perte, et entraînez-le sur `y_train` avec les mêmes réglages. Comparez sa perte et son accuracy en test à celles du premier modèle : que change le format des étiquettes ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- le même `keras.Sequential` que `model` : un nouveau modèle, aux poids neufs
- `loss="sparse_categorical_crossentropy"`, puis `fit` sur `y_train`, et `evaluate(model2)`, sans `one_hot`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model2 = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model2.compile(optimizer="sgd", loss=..., metrics=["accuracy"])
model2.fit(X_train, ..., batch_size=128, epochs=10, validation_split=0.2)
evaluate(model2)
```

### Solution

In [ ]:
model2 = keras.Sequential(
  [keras.Input(shape=(input_dim,)), layers.Dense(nb_classes, activation="softmax")]
)
model2.compile(
  optimizer="sgd", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
model2.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(model2)

# Les mêmes résultats, aux tirages aléatoires près (initialisation, ordre des batchs) :
# seul le format des cibles change. Les étiquettes entières évitent de stocker
# 10 valeurs par exemple.

## Perceptron multicouche régularisé

Écrivez une fonction `build_deep_model` qui renvoie un perceptron multicouche neuf, non compilé :

- 4 couches cachées de 20 neurones, avec l'activation ReLU ;
- sur leurs poids, une pénalité L2 de coefficient $10^{-4}$ (`kernel_regularizer`) et une initialisation orthogonale (`kernel_initializer`) ;
- la même couche de sortie softmax que la régression logistique.

Créez-en un, `deep_model`, affichez son résumé, compilez-le avec l'optimiseur `"adam"`, la perte `sparse_categorical_crossentropy` et l'accuracy, puis entraînez-le avec les mêmes réglages que la régression logistique. Combien de points d'accuracy en test gagne-t-il sur elle ? Pour combien de paramètres de plus ?

Les parties suivantes réutilisent `build_deep_model`.

**Documentation utile**

- [`keras.initializers.Orthogonal`](https://keras.io/api/layers/initializers/#orthogonal-class)

In [ ]:
def build_deep_model() -> keras.Model: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `kernel_regularizer=keras.regularizers.L2(1e-4)` ajoute à la perte $10^{-4} \sum w^2$, sur les poids de la couche ; les biais ne sont pas pénalisés
- `kernel_initializer="orthogonal"` : des matrices de poids initiales orthogonales, qui conservent la norme du signal d'une couche à l'autre
- Les quatre couches cachées ont les mêmes options : un dictionnaire `hidden`, passé avec `**hidden`, évite de les répéter
- Résultat attendu : 17 170 paramètres, environ 95 % d'accuracy en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_deep_model() -> keras.Model:
  hidden = dict(activation=..., kernel_regularizer=..., kernel_initializer=...)
  return keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(20, **hidden),
      ...,
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )


deep_model = build_deep_model()
deep_model.summary()
deep_model.compile(optimizer=..., loss=..., metrics=["accuracy"])
deep_model.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(deep_model)
```

### Solution

In [ ]:
def build_deep_model() -> keras.Model:
  """Un perceptron multicouche neuf : 4 couches cachées régularisées de 20 neurones."""
  hidden = dict(
    activation="relu",
    kernel_regularizer=keras.regularizers.L2(1e-4),
    kernel_initializer="orthogonal",
  )
  return keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(20, **hidden),
      layers.Dense(nb_classes, activation="softmax"),
    ],
    name="deep_model",
  )


deep_model = build_deep_model()
deep_model.summary()
deep_model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
deep_model.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evaluate(deep_model)

# Environ 6 points de plus que la régression logistique (95 % contre 89 %), pour
# 17 170 paramètres au lieu de 7 850. Une partie du gain vient d'Adam : la partie
# TensorBoard compare les deux optimiseurs sur ce modèle.

## Recherche d'hyper-paramètres

Combien de neurones dans la couche cachée, quel pas d'apprentissage ? KerasTuner essaie des combinaisons d'hyper-paramètres et garde la meilleure, notée sur le jeu de validation.

Écrivez une fonction `build_model(hp)` qui renvoie un perceptron à une couche cachée ReLU, compilé avec Adam, la perte `sparse_categorical_crossentropy` et l'accuracy. Elle lit ses hyper-paramètres dans `hp` :

- `units`, le nombre de neurones cachés : de 32 à 512, par pas de 32 ;
- `learning_rate`, le pas d'apprentissage d'Adam : de $10^{-4}$ à $10^{-2}$, en échelle logarithmique.

Lancez ensuite la recherche avec `keras_tuner.BayesianOptimization` : contrairement à `RandomSearch`, qui tire chaque essai au hasard, il choisit le suivant d'après les résultats des précédents. Objectif `val_accuracy`, 5 essais, chacun entraîné 3 fois pour une moyenne moins bruitée, 10 epochs par entraînement, des batchs de 1 500 exemples et 20 % de validation. Affichez enfin le résumé des essais, puis le meilleur modèle, avec sa perte et son accuracy en test.

**Documentation utile**

- [`keras_tuner.HyperParameters`](https://keras.io/keras_tuner/api/hyperparameters/)
- [`keras_tuner.BayesianOptimization`](https://keras.io/keras_tuner/api/tuners/bayesian/#bayesianoptimization-class)

In [ ]:
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `hp.Int("units", min_value=32, max_value=512, step=32)` et `hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")` renvoient la valeur de l'essai en cours
- `keras_tuner.BayesianOptimization(build_model, objective="val_accuracy", max_trials=5, executions_per_trial=3, seed=0, directory="tuner", project_name="mnist", overwrite=True)` : `overwrite=True` repart de zéro au lieu de reprendre la recherche déjà enregistrée dans `tuner/mnist`
- `tuner.search(...)` prend les arguments de `fit`
- `tuner.results_summary()` affiche les essais, du meilleur au moins bon ; `tuner.get_best_models()[0]` renvoie le meilleur modèle, déjà entraîné

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model:
  units = hp.Int("units", ...)
  learning_rate = hp.Float("learning_rate", ...)
  model = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(..., activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(...),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model


tuner = keras_tuner.BayesianOptimization(build_model, ...)
tuner.search(X_train, y_train, ...)
tuner.results_summary()
best_model = tuner.get_best_models()[0]
best_model.summary()
evaluate(best_model)
```

### Solution

In [ ]:
def build_model(hp: keras_tuner.HyperParameters) -> keras.Model:
  # Chaque appel de hp lit la valeur que le tuner a choisie pour l'essai en cours
  units = hp.Int("units", min_value=32, max_value=512, step=32)
  learning_rate = hp.Float(
    "learning_rate", min_value=1e-4, max_value=1e-2, sampling="log"
  )
  model = keras.Sequential(
    [
      keras.Input(shape=(input_dim,)),
      layers.Dense(units, activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  model.compile(
    optimizer=keras.optimizers.Adam(learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return model


tuner = keras_tuner.BayesianOptimization(
  build_model,
  objective="val_accuracy",
  max_trials=5,
  executions_per_trial=3,
  seed=0,
  directory="tuner",
  project_name="mnist",
  overwrite=True,
)
tuner.search_space_summary()

In [ ]:
tuner.search(X_train, y_train, epochs=10, batch_size=1500, validation_split=0.2)

In [ ]:
tuner.results_summary()
best_model = tuner.get_best_models()[0]
best_model.summary()
evaluate(best_model)

## Adam contre SGD sous TensorBoard

TensorBoard affiche les courbes d'apprentissage qu'un callback `keras.callbacks.TensorBoard`, passé à `fit`, écrit dans un dossier : un sous-dossier par entraînement. Lancez la cellule suivante avant l'exercice : le tableau de bord se remplit au fil des epochs (bouton de rechargement en haut à droite).

Entraînez ensuite deux perceptrons multicouches neufs (`build_deep_model()`) avec les réglages de la partie « Perceptron multicouche régularisé », l'un avec l'optimiseur `"adam"`, l'autre avec `"sgd"`. Passez à chaque `fit` un callback TensorBoard qui écrit dans `logs/adam` pour le premier, dans `logs/sgd` pour le second.

Comparez les courbes `epoch_loss` et `epoch_accuracy` des deux entraînements : lequel converge le plus vite ? Pourquoi ?

**Documentation utile**

- [`keras.callbacks.TensorBoard`](https://keras.io/api/callbacks/tensorboard/)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.callbacks.TensorBoard(log_dir="logs/adam")`, passé par `fit(..., callbacks=[...])`
- Un modèle neuf par optimiseur, `build_deep_model()` puis `compile` : sinon le second entraînement repart des poids du premier
- Dans TensorBoard, l'onglet *Scalars* superpose les courbes des deux dossiers ; `train` et `validation` séparent les courbes d'entraînement et de validation
- Relancer un entraînement ajoute ses courbes au même dossier : supprimez `logs` ou changez de nom de dossier pour repartir d'un tableau de bord propre

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for optimizer in ["adam", "sgd"]:
  model = build_deep_model()
  model.compile(
    optimizer=..., loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=...)
  model.fit(X_train, y_train, ..., callbacks=[...])
```

### Solution

In [ ]:
for optimizer in ["adam", "sgd"]:
  model = build_deep_model()
  model.compile(
    optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=f"logs/{optimizer}")
  model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.2,
    callbacks=[tensorboard],
  )

# Adam converge bien plus vite : il dépasse 90 % d'accuracy en validation dès la
# première epoch, SGD seulement après plusieurs. SGD fait un pas de même taille (0,01)
# pour tous les poids ; Adam adapte le pas de chaque poids à la taille de ses
# gradients récents, et garde une inertie d'un pas à l'autre (la moyenne des
# gradients).

## Arrêt précoce et sauvegarde du meilleur modèle

Plutôt que de fixer le nombre d'epochs à l'avance, on arrête l'entraînement quand la perte de validation ne baisse plus, et on garde le meilleur modèle vu.

Entraînez un perceptron multicouche neuf avec Adam, sur 50 epochs au plus, avec deux callbacks :

- `keras.callbacks.EarlyStopping` : arrêt après 3 epochs sans baisse de `val_loss`, et retour aux poids de la meilleure epoch ;
- `keras.callbacks.ModelCheckpoint` : sauvegarde du modèle dans `best.keras`, chaque fois que `val_loss` atteint un nouveau minimum.

Affichez le nombre d'epochs effectuées. Rechargez ensuite le modèle sauvegardé avec `keras.models.load_model`, et comparez son évaluation sur le test à celle du modèle en mémoire : pourquoi sont-elles identiques ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)`
- `keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss", save_best_only=True)`
- `fit(..., epochs=50, callbacks=[...])` renvoie un `History` : `len(history.history["loss"])` est le nombre d'epochs effectuées
- `keras.models.load_model("best.keras")` renvoie un modèle compilé, prêt pour `evaluate`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
model = build_deep_model()
model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
callbacks = [
  keras.callbacks.EarlyStopping(...),
  keras.callbacks.ModelCheckpoint(...),
]
history = model.fit(X_train, y_train, ..., callbacks=callbacks)
print(len(...))
evaluate(model)
evaluate(keras.models.load_model(...))
```

### Solution

In [ ]:
model = build_deep_model()
model.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
callbacks = [
  # Arrêt après 3 epochs sans baisse de val_loss, poids de la meilleure epoch remis
  keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
  ),
  # Sauvegarde du modèle à chaque nouveau minimum de val_loss
  keras.callbacks.ModelCheckpoint(
    "best.keras", monitor="val_loss", save_best_only=True
  ),
]
history = model.fit(
  X_train,
  y_train,
  batch_size=128,
  epochs=50,
  validation_split=0.2,
  callbacks=callbacks,
)
print(f"Epochs effectuées : {len(history.history['loss'])}")

evaluate(model)
best_model = keras.models.load_model("best.keras")
evaluate(best_model)

# Les mêmes chiffres : restore_best_weights remet dans le modèle les poids de la
# meilleure epoch, celle que ModelCheckpoint a sauvegardée en dernier. Sans lui, le
# modèle en mémoire garderait les poids de la dernière epoch, 3 epochs après la
# meilleure.